# 309 — Phenotype Sensitivity Analysis

Evaluate the sensitivity of the primary pharmacological phenotype to alternative response metrics and aggregation strategies. The primary phenotype is `selected_phenotype` from Notebook 308, defined as the median `LN_IC50` across compounds tested for each model.

## Inputs

- `308_analysis_cohort.csv` supplies the harmonized cohort, lineage annotations, and primary phenotype.
- `307_drug_specific_phenotypes.parquet` supplies the model–drug responses used to calculate five alternative model-level phenotypes.
- `306_gdsc_drug_response.csv` supplies the records used to calculate drug coverage per model.

## Analysis

Compare the primary and alternative phenotypes using Spearman concordance before and after centering within lineage. Estimate uncertainty by resampling models within lineages and assess the association between the primary phenotype and drug coverage.

## Output

`309_phenotype_sensitivity_representations.parquet` contains one row per model, with the primary and alternative phenotypes, their lineage-centered values, and drug coverage. Notebook 311 reads this file. The sensitivity summary is displayed in this notebook.

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pancancer_epigenetics.utils.paths import (Paths, project_relative_path)

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

PHARMACOLOGY_DIR = Paths.pharmacology

PHENOTYPE_PATH = (PHARMACOLOGY_DIR / "307_drug_specific_phenotypes.parquet")

COHORT_PATH = (PHARMACOLOGY_DIR / "308_analysis_cohort.csv")

DRUG_RESPONSE_PATH = (PHARMACOLOGY_DIR / "306_gdsc_drug_response.csv")

OUTPUT_DIR = PHARMACOLOGY_DIR

In [3]:
# =============================================================================
# Load phenotype-sensitivity inputs
# =============================================================================

drug_phenotypes = pd.read_parquet(PHENOTYPE_PATH)

analysis_cohort = pd.read_csv(COHORT_PATH)

drug_response = pd.read_csv(DRUG_RESPONSE_PATH)

print("Model–drug phenotypes:", drug_phenotypes.shape)
print("Analysis cohort :", analysis_cohort.shape)
print("Drug response   :", drug_response.shape)

Model–drug phenotypes: (180335, 6)
Analysis cohort : (712, 9)
Drug response   : (180335, 9)


In [4]:
# =============================================================================
# Construct model-level sensitivity representations
# =============================================================================

alternative_phenotypes = (
    drug_phenotypes.groupby("ModelID")
    .agg(
        mean_ln_ic50=("LN_IC50", "mean"),
        mean_auc=("AUC", "mean"),
        median_auc=("AUC", "median"),
        mean_z_score=("Z_SCORE", "mean"),
        median_z_score=("Z_SCORE", "median"),
    )
    .reset_index()
)

assert set(alternative_phenotypes["ModelID"]) == set(analysis_cohort["ModelID"]), (
    "ModelID sets differ between the 307 phenotypes and the 308 cohort."
)
assert set(drug_response["SANGER_MODEL_ID"]) == set(analysis_cohort["SangerModelID"]), (
    "Sanger model IDs differ between the 306 responses and the 308 cohort."
)

sensitivity_representations = analysis_cohort[
    ["ModelID", "SangerModelID", "OncotreeLineage", "selected_phenotype"]
].merge(
    alternative_phenotypes,
    on="ModelID",
    how="left",
    validate="one_to_one",
)

print("Sensitivity representations:", sensitivity_representations.shape)

Sensitivity representations: (712, 9)


In [5]:
# =============================================================================
# Compute model-level drug-response coverage
# =============================================================================

drug_coverage = (
    drug_response
    .groupby("SANGER_MODEL_ID")
    .agg(
        n_drug_response_measurements=("DRUG_ID", "size"),
        n_unique_drug_ids=("DRUG_ID", "nunique"),
    )
    .reset_index()
)

In [6]:
# =============================================================================
# Integrate drug-response coverage
# =============================================================================

sensitivity_representations = (
    sensitivity_representations
    .merge(
        drug_coverage,
        left_on="SangerModelID",
        right_on="SANGER_MODEL_ID",
        how="left",
    )
    .drop(columns="SANGER_MODEL_ID")
)

In [7]:
# =============================================================================
# Summarize model-level drug-response coverage
# =============================================================================

coverage_summary = (
    sensitivity_representations[
        [
            "n_drug_response_measurements",
            "n_unique_drug_ids",
        ]
    ]
    .describe()
    .loc[
        [
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

coverage_summary

,n_drug_response_measurements,n_unique_drug_ids
min,12.0,12.0
25%,251.0,251.0
50%,279.0,279.0
75%,280.0,280.0
max,295.0,295.0


In [8]:
# =============================================================================
# Compare drug-response coverage measures
# =============================================================================

coverage_measures_identical = (
    sensitivity_representations["n_drug_response_measurements"]
    .eq(sensitivity_representations["n_unique_drug_ids"])
    .all()
)

print("Coverage measures identical:", coverage_measures_identical)

Coverage measures identical: True


In [9]:
# =============================================================================
# Remove redundant coverage measure
# =============================================================================

sensitivity_representations = (
    sensitivity_representations
    .drop(columns="n_drug_response_measurements")
)

In [10]:
# =============================================================================
# Define phenotype sensitivity panel
# =============================================================================

PRIMARY_PHENOTYPE = "selected_phenotype"

CORE_SENSITIVITY_PHENOTYPES = [
    "mean_ln_ic50",
    "mean_auc",
    "mean_z_score",
]

SECONDARY_SENSITIVITY_PHENOTYPES = [
    "median_auc",
    "median_z_score",
]

PHENOTYPE_REPRESENTATIONS = [
    PRIMARY_PHENOTYPE,
    *CORE_SENSITIVITY_PHENOTYPES,
    *SECONDARY_SENSITIVITY_PHENOTYPES,
]

In [11]:
# =============================================================================
# Quantify model-level lineage-associated variance
# =============================================================================

lineage_eta_squared = {}

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    values = sensitivity_representations[phenotype_name]
    grand_mean = values.mean()

    total_ss = ((values - grand_mean) ** 2).sum()

    lineage_means = (
        sensitivity_representations
        .groupby("OncotreeLineage")[phenotype_name]
        .transform("mean")
    )

    between_ss = ((lineage_means - grand_mean) ** 2).sum()

    lineage_eta_squared[phenotype_name] = between_ss / total_ss

lineage_association_summary = (
    pd.Series(
        lineage_eta_squared,
        name="lineage_eta_squared",
    )
    .rename_axis("phenotype_representation")
    .reset_index()
)

lineage_association_summary

,phenotype_representation,lineage_eta_squared
0,selected_phenotype,0.393268
1,mean_ln_ic50,0.419249
2,mean_auc,0.368062
3,mean_z_score,0.422676
4,median_auc,0.189217
5,median_z_score,0.416973


In [12]:
# =============================================================================
# Construct lineage-adjusted phenotype representations
# =============================================================================

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    sensitivity_representations[
        f"{phenotype_name}_lineage_residual"
    ] = (
        sensitivity_representations[phenotype_name]
        - sensitivity_representations
        .groupby("OncotreeLineage")[phenotype_name]
        .transform("mean")
    )

In [13]:
# =============================================================================
# Compute raw phenotype concordance
# =============================================================================

raw_concordance = []

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    if phenotype_name == PRIMARY_PHENOTYPE:
        continue

    rho, _ = spearmanr(
        sensitivity_representations[PRIMARY_PHENOTYPE],
        sensitivity_representations[phenotype_name],
    )

    raw_concordance.append(
        {
            "phenotype_representation": phenotype_name,
            "raw_spearman_vs_primary": rho,
        }
    )

raw_concordance = pd.DataFrame(raw_concordance)

raw_concordance

,phenotype_representation,raw_spearman_vs_primary
0,mean_ln_ic50,0.988859
1,mean_auc,0.837572
2,mean_z_score,0.982204
3,median_auc,0.529384
4,median_z_score,0.980460


In [14]:
print(raw_concordance)

  phenotype_representation  raw_spearman_vs_primary
0             mean_ln_ic50                 0.988859
1                 mean_auc                 0.837572
2             mean_z_score                 0.982204
3               median_auc                 0.529384
4           median_z_score                 0.980460


In [15]:
# =============================================================================
# Compute lineage-adjusted phenotype concordance
# =============================================================================

lineage_adjusted_concordance = []

primary_residual = f"{PRIMARY_PHENOTYPE}_lineage_residual"

for phenotype_name in PHENOTYPE_REPRESENTATIONS:
    if phenotype_name == PRIMARY_PHENOTYPE:
        continue

    alternative_residual = f"{phenotype_name}_lineage_residual"

    rho, _ = spearmanr(
        sensitivity_representations[primary_residual],
        sensitivity_representations[alternative_residual],
    )

    lineage_adjusted_concordance.append(
        {
            "phenotype_representation": phenotype_name,
            "lineage_adjusted_spearman_vs_primary": rho,
        }
    )

lineage_adjusted_concordance = pd.DataFrame(
    lineage_adjusted_concordance
)

lineage_adjusted_concordance

,phenotype_representation,lineage_adjusted_spearman_vs_primary
0,mean_ln_ic50,0.982550
1,mean_auc,0.774524
2,mean_z_score,0.973637
3,median_auc,0.502197
4,median_z_score,0.969519


In [16]:
# =============================================================================
# Define lineage residualization
# =============================================================================

def residualize_by_lineage(data, phenotype_name):
    return (
        data[phenotype_name]
        - data
        .groupby("OncotreeLineage")[phenotype_name]
        .transform("mean")
    )

In [17]:
# =============================================================================
# Define lineage-stratified model bootstrap
# =============================================================================

N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 20260812

rng = np.random.default_rng(BOOTSTRAP_SEED)


def stratified_model_bootstrap(data, rng):
    sampled_lineages = []

    for _, lineage_data in data.groupby(
        "OncotreeLineage",
        sort=False,
    ):
        sampled_indices = rng.choice(
            lineage_data.index.to_numpy(),
            size=len(lineage_data),
            replace=True,
        )

        sampled_lineages.append(
            data.loc[sampled_indices]
        )

    return pd.concat(
        sampled_lineages,
        ignore_index=True,
    )

In [18]:
# =============================================================================
# Run lineage-stratified bootstrap
# =============================================================================

bootstrap_results = []

for bootstrap_iteration in range(N_BOOTSTRAP):
    bootstrap_sample = stratified_model_bootstrap(
        sensitivity_representations,
        rng,
    )

    primary_residual = residualize_by_lineage(
        bootstrap_sample,
        PRIMARY_PHENOTYPE,
    )

    for phenotype_name in PHENOTYPE_REPRESENTATIONS:
        if phenotype_name == PRIMARY_PHENOTYPE:
            continue

        raw_rho, _ = spearmanr(
            bootstrap_sample[PRIMARY_PHENOTYPE],
            bootstrap_sample[phenotype_name],
        )

        alternative_residual = residualize_by_lineage(
            bootstrap_sample,
            phenotype_name,
        )

        adjusted_rho, _ = spearmanr(
            primary_residual,
            alternative_residual,
        )

        bootstrap_results.append(
            {
                "bootstrap_iteration": bootstrap_iteration,
                "phenotype_representation": phenotype_name,
                "raw_spearman": raw_rho,
                "lineage_adjusted_spearman": adjusted_rho,
            }
        )

bootstrap_results = pd.DataFrame(bootstrap_results)

In [19]:
# =============================================================================
# Summarize bootstrap concordance uncertainty
# =============================================================================

bootstrap_summary = (
    bootstrap_results
    .groupby("phenotype_representation")
    .agg(
        raw_spearman_bootstrap_median=(
            "raw_spearman",
            "median",
        ),
        raw_spearman_ci_lower=(
            "raw_spearman",
            lambda x: x.quantile(0.025),
        ),
        raw_spearman_ci_upper=(
            "raw_spearman",
            lambda x: x.quantile(0.975),
        ),
        lineage_adjusted_bootstrap_median=(
            "lineage_adjusted_spearman",
            "median",
        ),
        lineage_adjusted_ci_lower=(
            "lineage_adjusted_spearman",
            lambda x: x.quantile(0.025),
        ),
        lineage_adjusted_ci_upper=(
            "lineage_adjusted_spearman",
            lambda x: x.quantile(0.975),
        ),
    )
    .reset_index()
)

bootstrap_summary

,phenotype_representation,raw_spearman_bootstrap_median,raw_spearman_ci_lower,raw_spearman_ci_upper,lineage_adjusted_bootstrap_median,lineage_adjusted_ci_lower,lineage_adjusted_ci_upper
0,mean_auc,0.837521,0.810791,0.860035,0.772352,0.736148,0.805307
1,mean_ln_ic50,0.988640,0.986247,0.990443,0.982013,0.977674,0.985076
2,mean_z_score,0.982285,0.974132,0.986959,0.973429,0.963321,0.979343
3,median_auc,0.529914,0.472494,0.585190,0.499733,0.434947,0.557916
4,median_z_score,0.980513,0.973422,0.985142,0.969193,0.960368,0.975223


In [20]:
# =============================================================================
# Assess drug-coverage sensitivity of the primary phenotype
# =============================================================================

coverage_lineage_residual = residualize_by_lineage(
    sensitivity_representations,
    "n_unique_drug_ids",
)

raw_coverage_rho, _ = spearmanr(
    sensitivity_representations[PRIMARY_PHENOTYPE],
    sensitivity_representations["n_unique_drug_ids"],
)

adjusted_coverage_rho, _ = spearmanr(
    sensitivity_representations[
        f"{PRIMARY_PHENOTYPE}_lineage_residual"
    ],
    coverage_lineage_residual,
)

coverage_sensitivity = pd.DataFrame(
    {
        "comparison": [
            "raw",
            "lineage_adjusted",
        ],
        "spearman_rho": [
            raw_coverage_rho,
            adjusted_coverage_rho,
        ],
    }
)

coverage_sensitivity

,comparison,spearman_rho
0,raw,0.241433
1,lineage_adjusted,0.146426


In [21]:
# =============================================================================
# Bootstrap drug-coverage sensitivity
# =============================================================================

coverage_bootstrap_results = []
coverage_rng = np.random.default_rng(BOOTSTRAP_SEED + 1)

for bootstrap_iteration in range(N_BOOTSTRAP):
    bootstrap_sample = stratified_model_bootstrap(
        sensitivity_representations,
        coverage_rng,
    )

    primary_residual = residualize_by_lineage(
        bootstrap_sample,
        PRIMARY_PHENOTYPE,
    )

    coverage_residual = residualize_by_lineage(
        bootstrap_sample,
        "n_unique_drug_ids",
    )

    raw_rho, _ = spearmanr(
        bootstrap_sample[PRIMARY_PHENOTYPE],
        bootstrap_sample["n_unique_drug_ids"],
    )

    adjusted_rho, _ = spearmanr(
        primary_residual,
        coverage_residual,
    )

    coverage_bootstrap_results.append(
        {
            "bootstrap_iteration": bootstrap_iteration,
            "raw_spearman": raw_rho,
            "lineage_adjusted_spearman": adjusted_rho,
        }
    )

coverage_bootstrap_results = pd.DataFrame(
    coverage_bootstrap_results
)

In [22]:
# =============================================================================
# Summarize drug-coverage bootstrap uncertainty
# =============================================================================

coverage_bootstrap_summary = pd.DataFrame(
    {
        "comparison": [
            "raw",
            "lineage_adjusted",
        ],
        "bootstrap_median": [
            coverage_bootstrap_results["raw_spearman"].median(),
            coverage_bootstrap_results[
                "lineage_adjusted_spearman"
            ].median(),
        ],
        "ci_lower": [
            coverage_bootstrap_results[
                "raw_spearman"
            ].quantile(0.025),
            coverage_bootstrap_results[
                "lineage_adjusted_spearman"
            ].quantile(0.025),
        ],
        "ci_upper": [
            coverage_bootstrap_results[
                "raw_spearman"
            ].quantile(0.975),
            coverage_bootstrap_results[
                "lineage_adjusted_spearman"
            ].quantile(0.975),
        ],
    }
)

coverage_bootstrap_summary

,comparison,bootstrap_median,ci_lower,ci_upper
0,raw,0.242110,0.174524,0.307214
1,lineage_adjusted,0.147063,0.074087,0.220321


In [23]:
# =============================================================================
# Assemble phenotype sensitivity summary
# =============================================================================

phenotype_sensitivity_summary = (
    lineage_association_summary
    .merge(
        raw_concordance,
        on="phenotype_representation",
        how="left",
    )
    .merge(
        lineage_adjusted_concordance,
        on="phenotype_representation",
        how="left",
    )
    .merge(
        bootstrap_summary,
        on="phenotype_representation",
        how="left",
    )
)

phenotype_sensitivity_summary.insert(
    1,
    "analysis_role",
    phenotype_sensitivity_summary["phenotype_representation"].map(
        {
            PRIMARY_PHENOTYPE: "primary",
            **{
                phenotype_name: "core_sensitivity"
                for phenotype_name in CORE_SENSITIVITY_PHENOTYPES
            },
            **{
                phenotype_name: "secondary_sensitivity"
                for phenotype_name in SECONDARY_SENSITIVITY_PHENOTYPES
            },
        }
    ),
)

phenotype_sensitivity_summary

,phenotype_representation,analysis_role,lineage_eta_squared,raw_spearman_vs_primary,lineage_adjusted_spearman_vs_primary,raw_spearman_bootstrap_median,raw_spearman_ci_lower,raw_spearman_ci_upper,lineage_adjusted_bootstrap_median,lineage_adjusted_ci_lower,lineage_adjusted_ci_upper
0,selected_phenotype,primary,0.393268,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mean_ln_ic50,core_sensitivity,0.419249,0.988859,0.982550,0.988640,0.986247,0.990443,0.982013,0.977674,0.985076
2,mean_auc,core_sensitivity,0.368062,0.837572,0.774524,0.837521,0.810791,0.860035,0.772352,0.736148,0.805307
3,mean_z_score,core_sensitivity,0.422676,0.982204,0.973637,0.982285,0.974132,0.986959,0.973429,0.963321,0.979343
4,median_auc,secondary_sensitivity,0.189217,0.529384,0.502197,0.529914,0.472494,0.585190,0.499733,0.434947,0.557916
5,median_z_score,secondary_sensitivity,0.416973,0.980460,0.969519,0.980513,0.973422,0.985142,0.969193,0.960368,0.975223


In [24]:
# =============================================================================
# Integrate drug-coverage sensitivity into final summary
# =============================================================================

sensitivity_summary = (
    phenotype_sensitivity_summary
    .rename(
        columns={
            "phenotype_representation": "analysis_target",
        }
    )
)

coverage_summary_row = pd.DataFrame(
    {
        "analysis_target": ["n_unique_drug_ids"],
        "analysis_role": ["technical_sensitivity"],
        "lineage_eta_squared": [np.nan],
        "raw_spearman_vs_primary": [raw_coverage_rho],
        "lineage_adjusted_spearman_vs_primary": [
            adjusted_coverage_rho
        ],
        "raw_spearman_bootstrap_median": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"] == "raw",
                "bootstrap_median",
            ].iloc[0]
        ],
        "raw_spearman_ci_lower": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"] == "raw",
                "ci_lower",
            ].iloc[0]
        ],
        "raw_spearman_ci_upper": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"] == "raw",
                "ci_upper",
            ].iloc[0]
        ],
        "lineage_adjusted_bootstrap_median": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"]
                == "lineage_adjusted",
                "bootstrap_median",
            ].iloc[0]
        ],
        "lineage_adjusted_ci_lower": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"]
                == "lineage_adjusted",
                "ci_lower",
            ].iloc[0]
        ],
        "lineage_adjusted_ci_upper": [
            coverage_bootstrap_summary.loc[
                coverage_bootstrap_summary["comparison"]
                == "lineage_adjusted",
                "ci_upper",
            ].iloc[0]
        ],
    }
)

sensitivity_summary = pd.concat(
    [
        sensitivity_summary,
        coverage_summary_row,
    ],
    ignore_index=True,
)

sensitivity_summary

,analysis_target,analysis_role,lineage_eta_squared,raw_spearman_vs_primary,lineage_adjusted_spearman_vs_primary,raw_spearman_bootstrap_median,raw_spearman_ci_lower,raw_spearman_ci_upper,lineage_adjusted_bootstrap_median,lineage_adjusted_ci_lower,lineage_adjusted_ci_upper
0,selected_phenotype,primary,0.393268,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,mean_ln_ic50,core_sensitivity,0.419249,0.988859,0.982550,0.988640,0.986247,0.990443,0.982013,0.977674,0.985076
2,mean_auc,core_sensitivity,0.368062,0.837572,0.774524,0.837521,0.810791,0.860035,0.772352,0.736148,0.805307
3,mean_z_score,core_sensitivity,0.422676,0.982204,0.973637,0.982285,0.974132,0.986959,0.973429,0.963321,0.979343
4,median_auc,secondary_sensitivity,0.189217,0.529384,0.502197,0.529914,0.472494,0.585190,0.499733,0.434947,0.557916
5,median_z_score,secondary_sensitivity,0.416973,0.980460,0.969519,0.980513,0.973422,0.985142,0.969193,0.960368,0.975223
6,n_unique_drug_ids,technical_sensitivity,NaN,0.241433,0.146426,0.242110,0.174524,0.307214,0.147063,0.074087,0.220321


In [25]:
# =============================================================================
# Save phenotype sensitivity artifacts
# =============================================================================

REPRESENTATIONS_OUTPUT_PATH = (
    OUTPUT_DIR / "309_phenotype_sensitivity_representations.parquet"
)

sensitivity_representations.to_parquet(
    REPRESENTATIONS_OUTPUT_PATH,
    index=False,
)

print("Phenotype sensitivity artifacts written")
print(f"Representations: {project_relative_path(REPRESENTATIONS_OUTPUT_PATH)}")

Phenotype sensitivity artifacts written
Representations: data/interim/pharmacology/309_phenotype_sensitivity_representations.parquet


## Summary

Across 712 models, the primary phenotype showed high concordance after lineage centering with mean `LN_IC50` and both `Z_SCORE` summaries (Spearman ρ = 0.970–0.983). Concordance was lower for mean AUC (ρ = 0.775) and median AUC (ρ = 0.502). The association with drug coverage decreased from ρ = 0.241 to 0.146 after lineage centering.

The median `LN_IC50` from Notebook 308 remains the primary phenotype. `309_phenotype_sensitivity_representations.parquet` provides the phenotype representations and drug coverage used by Notebook 311.